In [3]:
import os
import re

corpus = ""
data_dir = "Data"
count = 0  # Limitante para pruebas

for filename in os.listdir(data_dir):
    if filename.endswith(".txt"):
        filepath = os.path.join(data_dir, filename)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                text = f.read()
                cleaned = re.sub(r'[^\w\s]', ' ', text)
                corpus += cleaned + "\n"
            count += 1
            if count >= 5:  # Limitante para pruebas
                break
        except Exception as e:
            print(f"Error leyendo {filename}: {e}")

print(f"Corpus cargado con {len(corpus)} caracteres.")

Corpus cargado con 858614 caracteres.


## Tokenización

In [4]:
# Tokenización usando split
tokens = corpus.split()
print(f"Número de tokens: {len(tokens)}")
print("Primeros 10 tokens:", tokens[:10])

Número de tokens: 142747
Primeros 10 tokens: ['produced', 'from', 'images', 'available', 'at', 'The', 'Internet', 'Archive', 'OLIVERIO', 'GIRONDO']


## Stemming 

In [5]:
from nltk.stem import PorterStemmer

# Inicializar el stemmer
stemmer = PorterStemmer()

# Aplicar stemming a los tokens
stemmed_tokens = [stemmer.stem(token) for token in tokens]

print(f"Número de tokens stemmed: {len(stemmed_tokens)}")
print("Primeros 10 tokens stemmed:", stemmed_tokens[:10])

Número de tokens stemmed: 142747
Primeros 10 tokens stemmed: ['produc', 'from', 'imag', 'avail', 'at', 'the', 'internet', 'archiv', 'oliverio', 'girondo']


## Vectorización con TF-IDF

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Preparar los documentos (en este caso, un solo documento con los tokens stemmed)
docs = [' '.join(stemmed_tokens)]

# Crear el vectorizador TF-IDF
vectorizer = TfidfVectorizer()

# Vectorizar los documentos
X = vectorizer.fit_transform(docs)

print(f"Matriz TF-IDF creada con forma: {X.shape}")
print("Primeras 10 características (palabras):", vectorizer.get_feature_names_out()[:10])

Matriz TF-IDF creada con forma: (1, 16123)
Primeras 10 características (palabras): ['000' '019' '020' '058' '059' '10' '100' '1000' '10000' '100th']


## DataFrame Corpus con Procesamiento

In [12]:
import pandas as pd
import numpy as np
from nltk.stem import PorterStemmer

# Función para procesar cada documento
def process_document(text):
    """Limpia, tokeniza y aplica stemming al texto"""
    # Limpiar caracteres especiales
    cleaned = re.sub(r'[^\w\s]', ' ', text)
    # Tokenizar
    tokens = cleaned.split()
    # Aplicar stemming
    stemmer_local = PorterStemmer()
    stemmed = [stemmer_local.stem(token) for token in tokens]
    # Retornar como texto unido
    return ' '.join(stemmed)

# Cargar documentos (cada archivo es un documento)
doc_data = []
doc_id = 0
count_docs = 0

for filename in sorted(os.listdir(data_dir)):
    if filename.endswith(".txt"):
        filepath = os.path.join(data_dir, filename)
        try:
            with open(filepath, 'r', encoding='utf-8') as f:
                raw_text = f.read()
                doc_data.append({
                    'doc_id': doc_id,
                    'filename': filename,
                    'raw': raw_text
                })
                doc_id += 1
                count_docs += 1
                if count_docs >= 5:  # Limitante para pruebas
                    break
        except Exception as e:
            print(f"Error leyendo {filename}: {e}")

# Crear DataFrame
df_corpus = pd.DataFrame(doc_data)

# Aplicar función de procesamiento a cada documento
df_corpus['processed'] = df_corpus['raw'].apply(process_document)

# Crear TF-IDF Vectorizer y aplicar fit_transform
vectorizer_corpus = TfidfVectorizer()
tfidf_matrix = vectorizer_corpus.fit_transform(df_corpus['processed'])

# Agregar vector TF-IDF a cada fila
df_corpus['tfidf'] = [tfidf_matrix[i].toarray()[0] for i in range(len(df_corpus))]

print("=" * 60)
print("DataFrame Corpus con Documentos")
print("=" * 60)
print(f"Número de documentos: {len(df_corpus)}")
print(f"Número de features (palabras únicas): {tfidf_matrix.shape[1]}\n")
print(df_corpus[['doc_id', 'filename']].to_string())
print(f"\nPrimeras 20 features: {vectorizer_corpus.get_feature_names_out()[:20]}")

DataFrame Corpus con Documentos
Número de documentos: 5
Número de features (palabras únicas): 16123

   doc_id                                     filename
0       0  20 poemas para ser leídos en el tranvía.txt
1       1                40 years  40 años  40 ans.txt
2       2                               7 de julio.txt
3       3                   A Doll's House  a play.txt
4       4                   A First Spanish Reader.txt

Primeras 20 features: ['000' '019' '020' '058' '059' '10' '100' '1000' '10000' '100th' '101'
 '102' '103' '104' '107' '10th' '11' '110' '1156' '12']


## Consultas con Query Vector

In [10]:
# Realizar una consulta sobre los documentos
query = "historia love"  # Ejemplo de consulta

# Procesar la query usando la misma función
query_processed = process_document(query)
query_vector = vectorizer_corpus.transform([query_processed])
query_vector_array = query_vector.toarray()[0]

# Calcular similitud coseno con cada documento
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(query_vector, tfidf_matrix)[0]

# Agregar similitud al dataframe
df_corpus['similarity'] = similarities

# Mostrar resultados
print("=" * 60)
print("Resultados de Búsqueda")
print("=" * 60)
print(f"Query original: '{query}'")
print(f"Query procesada: '{query_processed}'\n")

# Mostrar documentos ordenados por similitud
results = df_corpus[['doc_id', 'filename', 'similarity']].sort_values('similarity', ascending=False)
print("Documentos ordenados por similitud con la query:\n")
for idx, row in results.iterrows():
    print(f"Doc {row['doc_id']}: {row['filename']}")
    print(f"  → Similitud: {row['similarity']:.4f}\n")

Resultados de Búsqueda
Query original: 'historia love'
Query procesada: 'historia love'

Documentos ordenados por similitud con la query:

Doc 3: A Doll's House  a play.txt
  → Similitud: 0.0083

Doc 2: 7 de julio.txt
  → Similitud: 0.0024

Doc 4: A First Spanish Reader.txt
  → Similitud: 0.0022

Doc 1: 40 years  40 años  40 ans.txt
  → Similitud: 0.0004

Doc 0: 20 poemas para ser leídos en el tranvía.txt
  → Similitud: 0.0000

